# [Scorio Lite](https://huggingface.co/datasets/harimo/scorio-lite)

Scorio Lite contains 1,211,520 attempts from four model configurations. Each question was
sampled with 80 seeds. The dataset has five competition-math splits and a 3,600-question
sample from superGPQA.

The `meta-*` configs contain every attempt without the six token-level lists. Per-model
configs add those lists. This notebook shows both forms and then selects one 80-attempt
candidate pool.

## Loading the math data

In [7]:
from datasets import load_dataset
from IPython.display import display

repo_name = "harimo/scorio-lite"
task = "aime_2026"

meta_math = load_dataset(repo_name, "meta-math", split=task)
print(meta_math)

Dataset({
    features: ['task', 'model', 'model_key', 'data_id', 'seed', 'prompt', 'trigger', 'sampling', 'text', 'finish_reason', 'num_prompt_tokens', 'num_completion_tokens', 'ground_truth', 'schema_version', 'evalscope_extracted_answer', 'evalscope_is_correct', 'grading_status', 'grader_id', 'extracted_answer', 'extracted_answer_is_list', 'boxed_answer_raw', 'has_box', 'has_valid_box', 'num_boxes', 'box_status', 'boxed_is_correct', 'evalscope_extraction_method', 'cv3b_label', 'cv3b_label_meaning', 'cv3b_prob', 'cv3b_abc_A', 'cv3b_abc_B', 'cv3b_abc_C', 'cv3b_ctx_A', 'cv3b_ctx_B', 'cv3b_ctx_C', 'llmv_model', 'llmv_model_revision', 'llmv_problem_understanding_expected', 'llmv_reasoning_validity_expected', 'llmv_conclusion_support_expected', 'llmv_mean_score_token_entropy_nats', 'llmv_mean_scale_probability_mass_before_renormalization', 'llmv_n_fine_grained_scores', 'llmv_score_definition', 'logprob_sentinel', 'tokens'],
    num_rows: 9600
})


## All four model configurations

The meta config contains the fields needed for accuracy, ranking, pass@k, and verifier analysis.
Sorting by model, question, and seed makes the array layout reproducible.

In [8]:
records = (meta_math
           .select_columns(["model_key", "data_id", "seed", "evalscope_is_correct",
                            "finish_reason", "num_completion_tokens"])
           .to_pandas()
           .sort_values(["model_key", "data_id", "seed"]))

summary = (records.groupby("model_key")
           .agg(attempts=("seed", "size"),
                questions=("data_id", "nunique"),
                accuracy=("evalscope_is_correct", "mean"),
                mean_completion_tokens=("num_completion_tokens", "mean"),
                length_finishes=("finish_reason", lambda x: int((x == "length").sum()))))

display(summary.round(3))

,attempts,questions,accuracy,mean_completion_tokens,length_finishes
model_key,,,,,
Qwen3.6-35B-A3B,2400,30,0.922,31111.792,86
gpt-oss-20b_high,2400,30,0.876,32561.430,241
gpt-oss-20b_low,2400,30,0.425,1765.267,0
gpt-oss-20b_medium,2400,30,0.777,10221.803,5


## One model with token-level detail

A per-model config has the same rows as the meta config and adds prompt and completion
token strings, log probabilities, and vocabulary ranks. This example loads the AIME split
for the medium reasoning setting.

In [9]:
model_name = "gpt-oss-20b_medium"
full = load_dataset(repo_name, f"{model_name}-math", split=task)

print(full)
print("token fields:", sorted(full.features["tokens"]))

Dataset({
    features: ['task', 'model', 'model_key', 'data_id', 'seed', 'prompt', 'trigger', 'sampling', 'text', 'finish_reason', 'num_prompt_tokens', 'num_completion_tokens', 'ground_truth', 'schema_version', 'evalscope_extracted_answer', 'evalscope_is_correct', 'grading_status', 'grader_id', 'extracted_answer', 'extracted_answer_is_list', 'boxed_answer_raw', 'has_box', 'has_valid_box', 'num_boxes', 'box_status', 'boxed_is_correct', 'evalscope_extraction_method', 'cv3b_label', 'cv3b_label_meaning', 'cv3b_prob', 'cv3b_abc_A', 'cv3b_abc_B', 'cv3b_abc_C', 'cv3b_ctx_A', 'cv3b_ctx_B', 'cv3b_ctx_C', 'llmv_model', 'llmv_model_revision', 'llmv_problem_understanding_expected', 'llmv_reasoning_validity_expected', 'llmv_conclusion_support_expected', 'llmv_mean_score_token_entropy_nats', 'llmv_mean_scale_probability_mass_before_renormalization', 'llmv_n_fine_grained_scores', 'llmv_score_definition', 'logprob_sentinel', 'tokens'],
    num_rows: 2400
})
token fields: ['completion_avg_logprob', 'c

## One candidate pool

In [10]:
question_id = 24
pool = full.filter(lambda row: row["data_id"] == question_id).sort("seed")

assert len(pool) == 80
assert pool["seed"] == list(range(80))

print("attempts:", len(pool))
print("ground truth:", pool[0]["ground_truth"])
print("first eight extracted answers:", pool["extracted_answer"][:8])
print("first eight rule-based grades:", pool["evalscope_is_correct"][:8])

attempts: 80
ground truth: 850
first eight extracted answers: ['950', '850', '850', '850', '850', '850', '850', '850']
first eight rule-based grades: [0, 1, 1, 1, 1, 1, 1, 1]


## One attempt

`text` is the response used by both verifiers. For gpt-oss it contains the final
channel, while the completion token list also contains the hidden reasoning channel. See
the dataset card before treating these two representations as interchangeable.

In [11]:
attempt = pool[0]
tokens = attempt["tokens"]

print("prompt:")
print(attempt["prompt"][:500])
print("\nresponse:")
print(attempt["text"][:700])
print("\nfinish:", attempt["finish_reason"])
print("completion tokens:", attempt["num_completion_tokens"])
print("first 12 token strings:", tokens["completion_token_list"][:12])
print("first 12 token log probabilities:", tokens["completion_logprob_list"][:12])

prompt:
Let $\triangle ABC$ be a triangle with $D$ on $\overline{BC}$ such that $\overline{AD}$ bisects $\angle BAC.$ Let $\omega$ be the circle that passes through $A$ and is tangent to segment $\overline{BC}$ at $D.$ Let $E \neq A$ and $F \neq A$ be the intersections of $\omega$ with segments $\overline{AB}$ and $\overline{AC},$ respectively. Suppose that $AB = 200, AC = 225,$ and all of $AE, AF, BD,$ and $CD$ are positive integers. Find the sum of all possible values of $BC.$

response:
Let  

\[
B=(0,0),\qquad C=(c,0),\qquad D=(d,0)
\]

and let \(AD\) be the internal bisector of \(\angle BAC\).
Since  

\[
\frac{BD}{DC}=\frac{AB}{AC}=\frac{200}{225}=\frac{8}{9},
\]

we can write  

\[
BD=8k,\qquad DC=9k,\qquad BC=17k\qquad(k>0).      \tag{1}
\]

--------------------------------------------------------------------
### 1.  Coordinates of \(A\)

Put \(A=(x,h)\) with \(h>0\).
Then  

\[
AB^2=x^2+h^2=200^2,\qquad 
AC^2=(x-17k)^2+h^2=225^2.
\]

Subtracting gives  

\[
(x-17k)^2-x^2=225^2

## A superGPQA record

superGPQA has its own configs because it adds discipline, field, difficulty, and frozen
sample identifiers. Streaming lets us inspect a record without materializing the complete
1.15-million-row meta split.

In [12]:
meta_gpqa = load_dataset(repo_name, "meta-gpqa", split="super_gpqa", streaming=True)
gpqa_record = next(iter(meta_gpqa))

print("task:", gpqa_record["task"])
print("model:", gpqa_record["model_key"])
print("field:", gpqa_record["field"])
print("difficulty:", gpqa_record["difficulty"])
print("stage-local data_id:", gpqa_record["data_id"])
print("global full_data_id:", gpqa_record["full_data_id"])
print("uuid:", gpqa_record["uuid"])

Resolving data files:   0%|          | 0/144 [00:00<?, ?it/s]

task: super_gpqa
model: Qwen3.6-35B-A3B
field: Aeronautical and Astronautical Science and Technology
difficulty: middle
stage-local data_id: 0
global full_data_id: 0
uuid: 351a4176c19a4ad8b4aeb7f9ade9c7c4


Use `full_data_id`, `uuid`, `selection_hash`, or `(stage, data_id)` when joining
superGPQA records. Its `data_id` is only unique within a stage.